# PMXT source-time validation

Tests the data adapter separately from model fitting and trading profitability. Market state follows Polymarket source timestamps, assuming a timely hypothetical feed; PMXT receipt times remain provenance. The primary future execution scenario is 100 ms total delay and uncapped capital/shares.

The cached sample contains six receipt hours on 4 May 2026. The analysis uses 01:00?04:00 UTC, with warm-up and two hours of later receipt context. Default execution is local and reads saved results. See [the validation report](../docs/reports/pmxt_validation.md).

In [1]:
from pathlib import Path
import sys,json
import pandas as pd
from IPython.display import display
PROJECT=Path.cwd()
if PROJECT.name=='notebooks': PROJECT=PROJECT.parent
sys.path.insert(0,str(PROJECT))
from src.data.pmxt_pilot import DEFAULT_ROOT,verify,load_plan
from src.data.pmxt_validation import validate
from src.common import sha256
ROOT=DEFAULT_ROOT
NAME='source_validation_v2'
OUTPUT=ROOT/NAME
RESULTS=PROJECT/'results/pmxt_pilot'/ROOT.name/NAME
REBUILD=False
if REBUILD:
    validate(ROOT,name=NAME)
summary=json.loads((RESULTS/'summary.json').read_text())
assert sha256(OUTPUT/'design.json')==summary['design_sha256']
_, plan=load_plan(ROOT)
assert sha256(ROOT/'plan.json')==json.loads((OUTPUT/'design.json').read_text())['universe_sha256']
for item in summary['inputs']:
    assert sha256(ROOT/item['path'])==item['sha256'],item['path']
for name,digest in summary['implementation_sha256'].items():
    assert sha256(PROJECT/name)==digest,name
for name,digest in summary.get('derived_sha256',{}).items():
    assert sha256(OUTPUT/name)==digest,name
for name,digest in summary.get('results_sha256',{}).items():
    assert sha256(RESULTS/name)==digest,name
display(verify(ROOT))

{'unchanged': True, 'files': 2913}

In [2]:
display(pd.Series({k:summary[k] for k in [
    'source_start','source_end','receipt_end','checkpoint_policy',
    'events','tokens','source_messages','event_hours','fully_observed_event_hours',
    'observed_event_time_fraction','complete_events_per_scan',
    'conditional_positive_event_hours','maximum_conditional_one_basket_gross_usd',
    'arrival_checks','historical_parameters_verified','model_schema_ready','net_pnl_estimated']}))
display(pd.read_csv(RESULTS/'event_hours.csv'))

source_start                                                        2026-05-04 01:00:00+00:00
source_end                                                          2026-05-04 04:00:00+00:00
receipt_end                                                         2026-05-04 06:00:00+00:00
checkpoint_policy                                                      authoritative_snapshot
events                                                                                     15
tokens                                                                                    200
source_messages                                                                       1188769
event_hours                                                                                45
fully_observed_event_hours                                                                  7
observed_event_time_fraction                                                         0.437162
complete_events_per_scan                    {'2026-05-04 01:

,event_id,event_slug,classification,hour,package_count,token_count,usable_token_fraction,known_package_fraction,clean_package_fraction,quoted_gap,...,minimum_ask_depth_shares,snapshot_conditional_gross_usd,max_source_age_seconds,fully_observed_seconds,any_package_observed_seconds,conditional_positive_seconds,conditional_peak_gross_usd,peak_source_at,label_conditional_gross,fee_net_executable_status
0,48297,databricks-ipo-closing-market-cap,negrisk,2026-05-04 01:00:00+00:00,1,8,0.750000,0.000000,0.000000,NaN,...,965.32,0,180.619,2707.461,2707.461,0.0,0.0,NaN,NaN,unknown_historical_parameters
1,106981,bitcoin-vs-gold-vs-sp-500-in-2026,negrisk,2026-05-04 01:00:00+00:00,1,3,1.000000,1.000000,1.000000,-0.050,...,611433.56,0,195.769,3600.000,3600.000,0.0,0.0,NaN,0.0,unknown_historical_parameters
2,230330,us-trade-deficit-in-2026,negrisk,2026-05-04 01:00:00+00:00,1,8,0.875000,0.000000,0.000000,NaN,...,273.19,0,34.817,1675.840,1675.840,0.0,0.0,NaN,NaN,unknown_historical_parameters
3,228259,how-many-tornadoes-in-the-us-in-2026,negrisk,2026-05-04 01:00:00+00:00,1,8,1.000000,1.000000,1.000000,-0.109,...,1107.94,0,66.725,2632.300,2632.300,0.0,0.0,NaN,NaN,unknown_historical_parameters
4,401356,colombia-presidential-election-1st-round-turnout,negrisk,2026-05-04 01:00:00+00:00,1,6,1.000000,1.000000,1.000000,-1.030,...,12844.39,0,15.720,3415.575,3415.575,0.0,0.0,NaN,NaN,unknown_historical_parameters
5,417522,chi1-edv-ccu-2026-05-22,negrisk,2026-05-04 01:00:00+00:00,1,3,1.000000,1.000000,1.000000,-0.990,...,1367.89,0,3353.678,3600.000,3600.000,0.0,0.0,NaN,0.0,unknown_historical_parameters
6,200252,openai-ipo-closing-market-cap-554,negrisk,2026-05-04 01:00:00+00:00,1,7,1.000000,1.000000,1.000000,-1.326,...,383.49,0,131.588,2824.000,2824.000,0.0,0.0,NaN,NaN,unknown_historical_parameters
7,201560,india-annual-inflation-2026,negrisk,2026-05-04 01:00:00+00:00,1,7,1.000000,1.000000,1.000000,-0.490,...,249.87,0,33.716,2515.809,2515.809,0.0,0.0,NaN,NaN,unknown_historical_parameters
8,439906,zelenskyy-of-tweets-may-5-may-12-2026,negrisk,2026-05-04 01:00:00+00:00,1,11,0.636364,0.000000,0.000000,NaN,...,263.24,0,15.489,425.147,425.147,0.0,0.0,NaN,NaN,unknown_historical_parameters
9,204610,of-seats-won-by-ppp-in-south-korea-by-elections,negrisk,2026-05-04 01:00:00+00:00,1,7,0.857143,0.000000,0.000000,NaN,...,16588.15,0,91.861,1801.127,1801.127,0.0,0.0,NaN,NaN,unknown_historical_parameters


In [3]:
checkpoints=pd.read_parquet(OUTPUT/'checkpoint_comparisons.parquet')
display(checkpoints[['exact','top_match','ask_exact','best_ask_size_match']].mean().rename('fraction'))
display(pd.read_csv(RESULTS/'token_quality.csv'))
# A full book can itself supply new state; disagreement is not a missing-data count.
display(pd.read_csv(RESULTS/'arrival_checks.csv'))

exact                  0.486465
top_match              0.951936
ask_exact              0.642514
best_ask_size_match    0.949088
Name: fraction, dtype: float64

,asset_id,price_change,book,repeated_snapshot_rows,checkpoint_comparisons,checkpoint_exact,checkpoint_top_match,top_mismatch_groups,checkpoint_disagreements,last_trade_price,crossed_groups,ambiguous_revision_groups,snapshot_delta_tie_conflicts,tick_size_change
0,1013303232772613078089364769282479364842719188...,8909,143,13,129,46,122,1215.0,83,9.0,121.0,NaN,NaN,NaN
1,1025205157286985031646867846747978044328792260...,5947,136,16,119,60,116,344.0,59,2.0,NaN,1.0,NaN,NaN
2,1028431895329686584841247243841691454310968435...,4513,105,1,103,13,98,18.0,90,1.0,NaN,NaN,NaN,NaN
3,1030857673646428597599568112983351459583344748...,7841,159,12,146,68,144,803.0,78,8.0,409.0,9.0,NaN,NaN
4,1031029027005737910334429611245959257202457300...,1173,212,72,139,94,139,NaN,45,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,9779779365142063138759332447581984349844925758...,1002,141,60,80,38,80,NaN,42,7.0,NaN,NaN,NaN,NaN
196,9841071128013801426343304957794320306607184788...,11380,94,1,92,25,84,619.0,67,NaN,NaN,1.0,NaN,NaN
197,9865378754846136456030025275933434374337253405...,493,251,165,85,45,85,NaN,40,3.0,NaN,NaN,NaN,NaN
198,9943734038648994121075062636841914198681847773...,5028,154,26,127,65,124,121.0,62,2.0,NaN,18.0,NaN,NaN


,event_id,package_id,decision_at,arrival_at,latency_ms,quantity,quoted_gross_usd,status,supported_legs,conditional_gross_usd


In [4]:
display(pd.Series({k:summary[k] for k in ['acceptance_targets','late_records_for_analysis',
    'recorded_transfer_bytes','raw_cache_bytes','limitations']}))
features=pd.read_parquet(OUTPUT/'hourly_features.parquet')
labels=pd.read_parquet(OUTPUT/'hourly_quote_labels.parquet')
assert not features.duplicated(['event_id','hour']).any()
assert not labels.duplicated(['event_id','hour']).any()
assert (features.hour<pd.Timestamp(summary['source_end'])).all()
assert (labels.fully_observed_seconds.between(0,3600+1e-8)).all()
assert labels.loc[labels.label_conditional_gross==0,'fully_observed_seconds'].ge(3600-1e-8).all()
print('Saved validation artifacts and label/feature invariants verified.')

acceptance_targets           {'complete_event_time_fraction': 0.95, 'fully_...
late_records_for_analysis                                                   42
recorded_transfer_bytes                                             2078388726
raw_cache_bytes                                                       13580838
limitations                  [Native full snapshot clocks assumed to identi...
dtype: object

Saved validation artifacts and label/feature invariants verified.


**Interpretation:** the archive needs to support coherent state reconstruction, adequate usable event-hours, valid basket metadata and historical trading parameters before a larger model study. Local consistency checks cannot prove completeness without sequence numbers or an independent feed. Quoted margins and arrival depth checks remain conditional diagnostics, not net P&L. No PMXT model or new final test is evaluated here.